# Data Preparation (CRISP-ML) – Classificação de tráfego IoT (MalwareFamily)

Dataset: *IoT Malware Network Traffic Dataset (CIC-YNU-2026)* – Kaggle.

Etapas (Módulo 4 do roadmap): **Data Cleaning → Famílias raras → Splitting → Missing Values → Outliers → Feature Scaling → Imbalanced Data**.

> Regra de ouro: tudo o que *aprende* com os dados (imputação, escala, pesos de classe) é ajustado **só no treino**, para evitar data leakage.

## 1. Carregar o dataset (diretamente do Kaggle)

In [ ]:
%pip install -q kagglehub scikit-learn pandas numpy

import glob, os
import numpy as np
import pandas as pd
import kagglehub

TARGET, GROUP, SEED = "MalwareFamily", "Hash", 42

path = kagglehub.dataset_download(
    "raneemababneh/iot-malware-network-traffic-dataset-cic-ynu-2026"
)
CSV_PATH = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)[0]
df = pd.read_csv(CSV_PATH)
print("Shape inicial:", df.shape)

## 2. Data Cleaning

- Os valores `inf` (existem na coluna `Rate`) passam a `NaN`, para serem tratados na imputação.
- Removem-se as linhas **duplicadas**.
- Saem das *features*: `Hash` (identifica a amostra, usa-se só para agrupar), `Arch` (sempre `arm`), `Tot size` e `LLC` (redundantes, iguais a `AVG` e `IPv`) e as colunas **constantes**.

In [ ]:
n_inf = int(np.isinf(df.select_dtypes("number")).sum().sum())
df = df.replace([np.inf, -np.inf], np.nan)
print("Valores inf convertidos em NaN:", n_inf)

n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicados removidos: {n_dup} | shape: {df.shape}")

num_cols = df.select_dtypes("number").columns
constantes = [c for c in num_cols if df[c].nunique(dropna=True) <= 1]
drop_cols = list(dict.fromkeys([TARGET, GROUP, "Arch", "Tot size", "LLC"] + constantes))
print("Colunas constantes:", constantes)

print("\nNaN por coluna:")
print(df.isna().sum()[lambda s: s > 0])

## 3. Famílias raras

O `Hash` identifica uma amostra e cada amostra gera muitas linhas. Como o split vai ser **por grupos** (hash inteiro no mesmo conjunto), uma família com poucos hashes não consegue ficar representada em treino, validação e teste.

Decisão: as famílias com menos de `MIN_HASHES` hashes são juntadas numa só classe, `Other_rare`.

In [ ]:
MIN_HASHES = 10

hashes_por_familia = df.groupby(TARGET)[GROUP].nunique().sort_values()
print(pd.concat([hashes_por_familia.rename("hashes"),
                 df[TARGET].value_counts().rename("linhas")], axis=1))

raras = hashes_por_familia[hashes_por_familia < MIN_HASHES].index.tolist()
print("\nFamílias raras juntadas em 'Other_rare':", raras)
df[TARGET] = df[TARGET].replace({r: "Other_rare" for r in raras})

## 4. Splitting Data (80 / 10 / 10)

`StratifiedGroupKFold` mantém as proporções das classes (**estratificado**) e nunca separa linhas do mesmo `Hash` por conjuntos diferentes (**grupos**). Faz-se em dois passos: 80 % treino / 20 % temporário, e depois o temporário divide-se em validação e teste.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

X = df.drop(columns=drop_cols)
y = df[TARGET]
groups = df[GROUP]

# 1.º passo: 80% treino / 20% temporário
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, temp_idx = next(sgkf.split(X, y, groups))
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_temp, y_temp, g_temp = X.iloc[temp_idx], y.iloc[temp_idx], groups.iloc[temp_idx]

# 2.º passo: 20% -> 10% validação / 10% teste
sgkf2 = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED)
val_idx, test_idx = next(sgkf2.split(X_temp, y_temp, g_temp))
X_val, y_val = X_temp.iloc[val_idx], y_temp.iloc[val_idx]
X_test, y_test = X_temp.iloc[test_idx], y_temp.iloc[test_idx]

print("Treino:", len(X_train), "| Validação:", len(X_val), "| Teste:", len(X_test))

In [ ]:
# Verificação 1: nenhum Hash aparece em dois conjuntos
g_train = set(groups.iloc[train_idx])
g_val = set(g_temp.iloc[val_idx])
g_test = set(g_temp.iloc[test_idx])
print("Hashes partilhados treino/val:", len(g_train & g_val))
print("Hashes partilhados treino/teste:", len(g_train & g_test))
print("Hashes partilhados val/teste:", len(g_val & g_test))

# Verificação 2: distribuição das classes em cada conjunto
dist = pd.concat([y_train.value_counts(), y_val.value_counts(), y_test.value_counts()],
                 axis=1, keys=["treino", "validação", "teste"]).fillna(0).astype(int)
dist

## 5. Missing Values, Outliers, Feature Scaling e Imbalanced Data

Tudo num único `Pipeline`, para ser **ajustado só no treino** e aplicado igual à validação e ao teste:

| Etapa | Técnica | Porquê |
|---|---|---|
| Outliers | `log1p` nas colunas muito enviesadas | Comprime valores extremos **sem apagar linhas** (em tráfego de rede, os extremos costumam ser ataques reais) |
| Missing Values | `SimpleImputer(median)` | A mediana é robusta a outliers |
| Feature Scaling | `StandardScaler` | Põe todas as colunas na mesma escala |
| Imbalanced Data | `class_weight="balanced"` | Dá mais peso às famílias pequenas, sem criar dados sintéticos (o SMOTE seria pesado com ~590 mil linhas) |

As colunas enviesadas escolhem-se **só com o treino** (|skewness| > 1, valores ≥ 0 e mais de 2 valores distintos).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier

skew_train = X_train.skew()
skewed = [c for c in X_train.columns
          if abs(skew_train[c]) > 1 and X_train[c].min() >= 0 and X_train[c].nunique() > 2]
print(f"{len(skewed)} colunas com log1p:", skewed)

log_transform = ColumnTransformer(
    [("log1p", FunctionTransformer(np.log1p), skewed)],
    remainder="passthrough",
)

N_TREES = 100  # reduz para 20-30 se o treino demorar muito

pipe = Pipeline([
    ("log", log_transform),                              # outliers
    ("imputer", SimpleImputer(strategy="median")),       # missing values
    ("scaler", StandardScaler()),                        # feature scaling
    ("model", RandomForestClassifier(                    # imbalanced data
        n_estimators=N_TREES, class_weight="balanced",
        random_state=SEED, n_jobs=-1)),
])

## 6. Verificação rápida na validação

Isto **não** é a avaliação final: serve só para confirmar que o pipeline corre de ponta a ponta. A avaliação a sério (matriz de confusão, tuning, etc.) fica para o Módulo 7. O teste só se usa no fim.

In [ ]:
from sklearn.metrics import f1_score, classification_report

pipe.fit(X_train, y_train)
pred_val = pipe.predict(X_val)

print("F1 macro (validação):", round(f1_score(y_val, pred_val, average="macro"), 4))
print(classification_report(y_val, pred_val, digits=3, zero_division=0))

## 7. Decisões para o relatório (CRISP-ML – Data Preparation)

- **Limpeza:** `inf` → `NaN`; duplicados removidos; removidas `Hash` (só agrupamento), `Arch`, `Tot size`, `LLC` e colunas constantes.
- **Famílias raras:** famílias com menos de 10 hashes juntadas em `Other_rare`, porque com split por grupos não conseguiriam estar em treino, validação e teste.
- **Split:** 80/10/10, estratificado e por grupos (`Hash`), com `random_state=42`. Sem hashes partilhados entre conjuntos.
- **Outliers:** `log1p` nas colunas enviesadas; nenhuma linha removida.
- **Missing values:** mediana, aprendida no treino.
- **Scaling:** `StandardScaler`, ajustado no treino.
- **Desequilíbrio:** `class_weight="balanced"`.
- **Métrica:** F1 macro (a accuracy engana com classes tão desequilibradas).